# WP4v2 — Notebook 1 : Génération de la base de données

Paires : `(mean_z_mae, cls_clip)`

- **Source** : `mean(z_i^MAE)` ∈ ℝ^1024 — justifié par le linear probing 73.5% de l'article MAE
- **Cible** : token CLS du ViT CLIP (vision tower LLaVA, index 0), L2-normalisé ∈ ℝ^1024

Le CLS CLIP est discriminant par construction (loss contrastive CLIP) et vit dans le même espace
que les tokens patch — ce qui permet de réutiliser le MLP connector LLaVA à l'inférence.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')


In [ ]:
# MAE
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()
print('MAE charge')

# Vision tower LLaVA uniquement (pas MLP connector, pas LLM)
llava = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
)
vision_tower = llava.vision_tower.to(DEVICE).eval()
del llava
torch.cuda.empty_cache()

clip_proc = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
print(f'Vision tower chargee — VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision import transforms

ds_train = load_dataset('parquet', data_files={
    'train': './imagenet100/data/train-*.parquet',
})
ds_val = load_dataset('parquet', data_files={
    'validation': './imagenet100/data/validation-*.parquet',
})

transform = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(336),
    transforms.ToTensor(),
])

class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.data      = hf_dataset
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item  = self.data[idx]
        image = item['image'].convert('RGB')
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

dataset_train = HFImageDataset(ds_train['train'],      transform=transform)
dataset_val   = HFImageDataset(ds_val['validation'],   transform=transform)

print(f'Train : {len(dataset_train)} | Val : {len(dataset_val)}')


In [ ]:
def generate_pairs(dataset, batch_size=64, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE=='cuda'))
    all_z_mae, all_cls_clip, all_labels = [], [], []

    for images_336, labels in tqdm(loader, desc=desc):
        B = images_336.shape[0]

        # MAE : resize 336->224, full encoding sans masquage
        images_224 = F.interpolate(images_336, size=(224,224), mode='bilinear', align_corners=False)
        mae_in = mae_processor(images=list(images_224), return_tensors='pt', do_rescale=False)
        mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}
        with torch.no_grad():
            out = mae_encoder(**mae_in, noise=torch.zeros(B, 196).to(DEVICE))
        z_mae = out.last_hidden_state[:, 1:].mean(dim=1).cpu().float()  # (B, 1024)

        # CLIP : token CLS (index 0) de la vision tower
        clip_in = clip_proc(images=list(images_336), return_tensors='pt', do_rescale=False)
        pix = clip_in['pixel_values'].to(DEVICE).half()
        with torch.no_grad():
            cls = vision_tower(pix).last_hidden_state[:, 0]  # (B, 1024)
        cls_clip = F.normalize(cls.cpu().float(), dim=-1)     # L2-normalise

        all_z_mae.append(z_mae)
        all_cls_clip.append(cls_clip)
        all_labels.append(labels)

    return {
        'z_mae':    torch.cat(all_z_mae),
        'cls_clip': torch.cat(all_cls_clip),
        'labels':   torch.cat(all_labels),
    }

data_train = generate_pairs(dataset_train, batch_size=64, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=64, desc='Val')

torch.save(data_train, 'wp4v2_pairs_train.pt')
torch.save(data_val,   'wp4v2_pairs_val.pt')
print(f'Sauvegarde OK — Train: {data_train["z_mae"].shape} | Val: {data_val["z_mae"].shape}')

# Verification discriminabilite des cibles
import numpy as np
idx5 = np.random.choice(len(data_val['labels']), 5, replace=False)
sim  = data_val['cls_clip'][idx5] @ data_val['cls_clip'][idx5].T
print('\nSimilarites cosinus cls_clip (5 images) :')
print(sim.numpy().round(4))

del vision_tower, mae_encoder
torch.cuda.empty_cache()
